## Phase 1.1 - Accessing Semantic Scholar and Handling IPs/API keys 

In [65]:
import os
import sys
from dotenv import load_dotenv
import requests
from semanticscholar import AsyncSemanticScholar
import asyncio
import nest_asyncio
import pandas as pd

In [66]:
nest_asyncio.apply()

load_dotenv(override=True)


# 1. GETTING USER API KEY. DEFAULTS AND CHECKS IP IF KEY IS NOT FOUND
api_key = os.getenv("SEMANTIC_SCHOLAR_API_KEY")
is_key_good = False

print("Searching for User's API Key...")

if api_key is None or api_key == "your_key_here" or api_key.strip() == "":
    print(f"User provided this: '{api_key}' as an API Key. Defaulting to lower limit access...")
    print("Checking IP status...")

    #CHECKING IF IP IS BLOCKED OR NOT
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    query_params = {"query": "test", "limit": 1}
    
    print("Pinging Semantic Scholar...")
    response = requests.get(url, params=query_params)

    if response.status_code == 200:
        print("Your IP is clear to access Semantic Scholar. Ready to start!")
    elif response.status_code == 429:
        sys.exit("Your IP is currently blocked from using Semantic Scholar")

else:
    print("API Key found! Going to Semantic Scholar...")
    is_key_good = True

Searching for User's API Key...
API Key found! Going to Semantic Scholar...


In [67]:

# 2. CONNECTING TO SEMANTIC SCHOLAR
sch = None
if is_key_good:
    sch = AsyncSemanticScholar(api_key=api_key)
else:
    sch = AsyncSemanticScholar()


# 3. VALIDATING THE API KEY
try:
    testing_api_key_works = await sch.get_paper("8ceb75144ecb846bf463e7565e6a18998ae29d1a", fields=["title"])
    # The id for Protein measurement with the Folin phenol reagent. That is the most cited paper on Semantic Scholar.
    print("Your API Key has been approved!")

except Exception as e:
    print(f"API Key Error: {e}")
    sys.exit("Your API key was not authorized for usage by Semantic Scholar")



Your API Key has been approved!


## Phase 1.2 - First Request

In [68]:
chills_papers = []
target_num = 500

topic_terms = [
    "frisson",
    "psychogenic",
    "aesthetic chill",
    "musical thrill",
    "music thrill",
    "musical chill",
    "music chill",
    "music-induced",
    "music induced",
    "music-evoke",
    "music evoke",
    "induced pleasure",
    "evoked emotion",
    "piloerection",
    "goosebump",
    "gooseflesh",
    "chills response",
    "tingl",
    "spine",
    "peak experien"
]

neuro_terms = [
    "neuro",
    #brain
    "brain scan",
    "nerv",
    "fmri",
    "eeg",
    #"scan",
    "dopamine",
    #reward
    "reward path",
    "reward circuit",
    "reward system",
    "cortex",
    "striatum",
    "autonom",
    "recept",
    "amygdala",
    "insula",
    "hemo",
    #"physiologic",
    "electroderma",
    "skin conduct",
    "galv",
    "gsr",
    "hrv",
    "nucl",
    "accumben",
    "ventr",
    "tegment",
    "vta",
    "limbic",
    "thalam",
    "sympathetic",
    "endogen",
    "opioid",
    "endorphin",
    "naltrexone",
    "pupil",
    "dilat",
    "respirat",
    "fnir"
]

search_queries = [
    "music frisson",
    "music chill",
    "music thrill",
    "aesthetic chill",
    "music goosebump",
    "music pleasure",
    "sound bump reaction",
    "psycho shiver",
    "music piloerect",
]



In [69]:
print("Beginning Extraction for Major Phenomena: Musical Frisson...")

results = []
ids = set()

for sq in search_queries:
    temp_results = await sch.search_paper(
        sq,
        limit=100,
        fields=[
            "title",
            "authors",
            "year",
            "citationCount",
            "abstract"
        ]
    )

    print(f"Found {temp_results.total} papers...")

    for paper in temp_results:
        unq = getattr(paper, "paperID", getattr(paper, "title", None))
        if unq not in ids:
            ids.add(unq)
            results.append(paper)

    print(f"Keeping {len(results)} papers...")
    await asyncio.sleep(1)


print(f"Success! Found {len(results)} total papers")

Beginning Extraction for Major Phenomena: Musical Frisson...
Found 571451 papers...
Keeping 995 papers...
Found 652083 papers...
Keeping 1246 papers...
Found 583943 papers...
Keeping 1471 papers...
Found 437049 papers...
Keeping 2420 papers...
Found 570166 papers...
Keeping 2461 papers...
Found 693921 papers...
Keeping 3421 papers...
Found 21263 papers...
Keeping 4409 papers...
Found 103659 papers...
Keeping 5389 papers...
Found 569968 papers...
Keeping 5419 papers...
Success! Found 5419 total papers


In [70]:
print(f"Beginning term filter on {len(results)} papers...")

filtered_papers = []
amount_missing_abstract = 0

rep_neuro_terms = []
rep_topic_terms = []

for paper in results:
    #Some papers may not have abstracts so this checks for it safely
    title = getattr(paper, "title", None)
    abstract = getattr(paper, "abstract", None)

    if abstract is None:
        amount_missing_abstract += 1
        continue #Skips to next if it doesn't have an abstract

    searchable_text = f"{title} {abstract}".lower()
    is_found_topic = False
    is_found_neuro = False

    temp_topic = []
    for t in topic_terms:
        if t.lower() in searchable_text:
            is_found_topic = True
            temp_topic.append(t.lower())
            #break

    temp_neuro = []
    for t in neuro_terms:
        if t.lower() in searchable_text:
            is_found_neuro = True
            temp_neuro.append(t.lower())
            #break -- To find accurate term frequencies, I won't use a break

    if is_found_topic == False or is_found_neuro == False:
        continue

    rep_topic_terms.extend(temp_topic)
    rep_neuro_terms.extend(temp_neuro)

    filtered_papers.append(paper)

print(f"Successs! Kept {len(filtered_papers)} total papers")
print(f"{amount_missing_abstract} papers were missing an abstract")
    

Beginning term filter on 5419 papers...
Successs! Kept 65 total papers
3951 papers were missing an abstract


In [71]:
#STORING THE FOUND TERMS FROM THE KEPT PAPERS INTO SORTED DICTIONARIES
topic_freq = {}
for t in rep_topic_terms:
    if t not in topic_freq:
        topic_freq[t] = 1
    else:
        topic_freq[t] += 1

topic_freq = dict(sorted(topic_freq.items(), key=lambda item: item[1], reverse=True))
print("Frequency of Topic Terms being found:")
for t,c in topic_freq.items():
    print(f"{t}: {c}")

neuro_freq = {}
for t in rep_neuro_terms:
    if t not in neuro_freq:
        neuro_freq[t] = 1
    else:
        neuro_freq[t] += 1
print()

neuro_freq = dict(sorted(neuro_freq.items(), key=lambda item: item[1], reverse=True))
print("Frequency of Neuroscience Terms being found:")
for t,c in neuro_freq.items():
    print(f"{t}: {c}")

Frequency of Topic Terms being found:
music-induced: 17
goosebump: 13
aesthetic chill: 12
frisson: 9
induced pleasure: 9
music-evoke: 8
tingl: 7
music chill: 6
peak experien: 5
musical chill: 5
piloerection: 4
evoked emotion: 2
psychogenic: 1
spine: 1
chills response: 1
music evoke: 1

Frequency of Neuroscience Terms being found:
neuro: 28
cortex: 12
autonom: 9
dopamine: 8
skin conduct: 8
fmri: 8
nucl: 8
eeg: 7
ventr: 7
electroderma: 7
striatum: 5
amygdala: 5
reward system: 5
accumben: 5
respirat: 4
insula: 3
opioid: 3
reward circuit: 3
pupil: 2
naltrexone: 2
nerv: 2
limbic: 2
hemo: 2
thalam: 1
dilat: 1
endogen: 1
fnir: 1
recept: 1
